# Phenix 线上 BigQuery 归因审计

## tl;dr

Phenix 标识在 Firebase 与 Origin H5 聚合中均未观察到；其他渠道可进入 Firebase 来源字段，Origin H5 实时表缺少 UTM 和会话键。

## Context & Methods

完整日窗口为 2026-08-27 至 2026-09-01；9 月 2 日 intraday 不参与比较。仅使用已执行的只读聚合快照。


In [ ]:
from pathlib import Path
import json
root = Path.cwd()
s = json.loads((root / 'analysis/phenix_online_attribution_audit_2026_09_02/channel_summary.json').read_text(encoding='utf-8'))
print(s['complete_day_window'], s['firebase_query_success_count'], s['firebase_query_count'])


## Data

SQL 与回执位于 `analysis/phenix_online_attribution_audit_2026_09_02/`。


In [ ]:
for k, v in sorted(s['firebase_dataset_summary'].items()):
    print(k, v['platforms'], v['days'], v['table_rows'], v['visible_coverage'], v['phx_markers'])


## Results

检查 H5 归因字段、会话键和 Phenix 标识。


In [ ]:
web = s['origin_realtime_web_summary']
print('Origin H5 events:', web['event_rows'])
print('UTM source rate:', web['utm_source_rate'])
print('Session key missing:', web['session_id_missing_rate'])
print('Phenix markers:', web['phx_markers'])
assert web['phx_markers'] == 0
assert web['utm_source'] == 0 and web['utm_medium'] == 0 and web['utm_campaign'] == 0


## Takeaways

先修复渠道映射、源表刷新和会话键，再判断 Phenix 人群质量与留存差异。空结果不等于业务值为 0。
